<a href="https://colab.research.google.com/github/Madankk-06/Deep-learning-projects/blob/main/5_Custom_Gradient_Descent_Optimizers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Import PyTorch for tensor operations.
# Automatic differentiation is disabled as required by the task.
import torch

# Import NumPy for numerical computations.
import numpy as np

# Set random seeds for reproducible normalization results.
torch.manual_seed(42)
np.random.seed(42)

In [2]:
# Implement Batch Normalization forward propagation.
# Batch statistics are computed across all samples
# to normalize feature distributions.

def batch_norm_forward(x, gamma, beta, eps=1e-5):

    # Compute batch mean.
    mean = torch.mean(x, dim=0, keepdim=True)

    # Compute batch variance.
    variance = torch.var(x, dim=0, unbiased=False, keepdim=True)

    # Normalize every feature using the batch statistics.
    x_hat = (x - mean) / torch.sqrt(variance + eps)

    # Scale normalized values using gamma.
    # Shift normalized values using beta.
    output = gamma * x_hat + beta

    return output, (x, x_hat, mean, variance, gamma, beta, eps)

In [3]:
# Compute manual gradients for Batch Normalization.
# Gradients are propagated through normalization,
# scaling (gamma), and shifting (beta).

def batch_norm_backward(dout, cache):

    x, x_hat, mean, variance, gamma, beta, eps = cache

    N = x.shape[0]

    # Gradient with respect to gamma parameter.
    dgamma = torch.sum(dout * x_hat, dim=0)

    # Gradient with respect to beta parameter.
    dbeta = torch.sum(dout, dim=0)

    # Gradient through scaling transformation.
    dx_hat = dout * gamma

    # Gradient through variance normalization.
    dvar = torch.sum(
        dx_hat * (x - mean) * (-0.5) * (variance + eps) ** (-1.5),
        dim=0
    )

    # Gradient through mean normalization.
    dmean = torch.sum(
        dx_hat * (-1 / torch.sqrt(variance + eps)),
        dim=0
    )

    dmean += dvar * torch.mean(-2 * (x - mean), dim=0)

    # Final gradient propagated to the input.
    dx = (
        dx_hat / torch.sqrt(variance + eps)
        + dvar * 2 * (x - mean) / N
        + dmean / N
    )

    return dx, dgamma, dbeta

In [4]:
# Implement Layer Normalization.
# Statistics are computed independently
# for every training sample.

def layer_norm_forward(x, gamma, beta, eps=1e-5):

    # Compute mean across features.
    mean = torch.mean(x, dim=1, keepdim=True)

    # Compute variance across features.
    variance = torch.var(x, dim=1, unbiased=False, keepdim=True)

    # Normalize every feature vector.
    x_hat = (x - mean) / torch.sqrt(variance + eps)

    # Apply learnable scaling and shifting.
    output = gamma * x_hat + beta

    return output, (x, x_hat, mean, variance, gamma, beta, eps)

In [5]:
# Compute manual gradients for Layer Normalization.
# Gradients pass through normalization,
# scaling, and shifting operations.

def layer_norm_backward(dout, cache):

    x, x_hat, mean, variance, gamma, beta, eps = cache

    D = x.shape[1]

    # Gradient with respect to gamma.
    dgamma = torch.sum(dout * x_hat, dim=0)

    # Gradient with respect to beta.
    dbeta = torch.sum(dout, dim=0)

    # Gradient through scaling operation.
    dx_hat = dout * gamma

    # Gradient through variance normalization.
    dvar = torch.sum(
        dx_hat * (x - mean) * (-0.5) * (variance + eps) ** (-1.5),
        dim=1,
        keepdim=True
    )

    # Gradient through mean normalization.
    dmean = torch.sum(
        dx_hat * (-1 / torch.sqrt(variance + eps)),
        dim=1,
        keepdim=True
    )

    dmean += dvar * torch.mean(-2 * (x - mean), dim=1, keepdim=True)

    # Final gradient propagated to the input.
    dx = (
        dx_hat / torch.sqrt(variance + eps)
        + dvar * 2 * (x - mean) / D
        + dmean / D
    )

    return dx, dgamma, dbeta

In [6]:
# Initialize the learnable scale (gamma)
# and shift (beta) parameters.

gamma = torch.ones(10)

beta = torch.zeros(10)

In [7]:
# Generate a batch of feature vectors
# for normalization experiments.

x = torch.randn(8,10)

In [8]:
# Normalize the batch using Batch Normalization.

batch_output, batch_cache = batch_norm_forward(
    x,
    gamma,
    beta
)

In [9]:
# Normalize each sample using Layer Normalization.

layer_output, layer_cache = layer_norm_forward(
    x,
    gamma,
    beta
)

In [10]:
# Simulate gradients received from the next layer.

dout = torch.randn_like(batch_output)

# Backpropagate through Batch Normalization.
dx_bn, dgamma_bn, dbeta_bn = batch_norm_backward(
    dout,
    batch_cache
)

# Backpropagate through Layer Normalization.
dx_ln, dgamma_ln, dbeta_ln = layer_norm_backward(
    dout,
    layer_cache
)

In [11]:
# Verify that the normalization layers
# preserve the input tensor dimensions.

print("BatchNorm Output Shape :", batch_output.shape)

print("LayerNorm Output Shape :", layer_output.shape)

BatchNorm Output Shape : torch.Size([8, 10])
LayerNorm Output Shape : torch.Size([8, 10])
